# Generative Models for Inverse Problems


## Bayesian Framing and Variational Reconstruction

The whole module can be read as a progressive build-up toward this point. We started from classical inverse problems, then moved to supervised neural reconstruction, then discussed noise and realism in the data-generation process, and finally introduced several kinds of generative models. The final question is therefore inevitable: how can these generative models actually be used to solve inverse problems in imaging?

This chapter is the conceptual synthesis of the course. It shows how a learned image prior can be combined with the measurement model to produce reconstructions that are not only visually plausible, but also informed by the physics of acquisition.


**Bayesian formulation.**

Let

$$
\boldsymbol{y}^\delta = K \boldsymbol{x}^\dagger + \boldsymbol{e}
$$

be the measurement model. If the noise is Gaussian,

$$
\boldsymbol{e} \sim \mathcal{N}(0,\sigma^2 I),
$$

then the likelihood is

$$
p(\boldsymbol{y}^\delta| \boldsymbol{x})
\propto
\exp\left(-\frac{1}{2\sigma^2}\|K\boldsymbol{x}-\boldsymbol{y}^\delta\|_2^2\right).
$$

If we also have a prior distribution $p(\boldsymbol{x})$ over plausible images, Bayes' theorem gives the posterior {cite}`kaipio2005statistical`

$$
p(\boldsymbol{x}| \boldsymbol{y}^\delta)
\propto
p(\boldsymbol{y}^\delta| \boldsymbol{x})p(\boldsymbol{x}).
$$

This formula is the bridge between classical regularization and modern generative reconstruction. The likelihood encodes **data consistency**, while the prior encodes what kinds of images are considered plausible.

The great promise of generative models is precisely that they can provide a powerful learned approximation of the prior term.

**MAP estimation and the classical variational form.**

If one seeks only a single reconstruction, a natural choice is the maximum a posteriori estimator:

$$
\widehat{\boldsymbol{x}}_{\mathrm{MAP}}
=
\operatorname*{arg\,max}_\boldsymbol{x} p(\boldsymbol{x}| \boldsymbol{y}^\delta).
$$

Taking the negative logarithm and dropping constants independent of $\boldsymbol{x}$, this becomes

$$
\widehat{\boldsymbol{x}}_{\mathrm{MAP}}
=
\operatorname*{arg\,min}_\boldsymbol{x}
\frac{1}{2\sigma^2}\|K\boldsymbol{x}-\boldsymbol{y}^\delta\|_2^2
-\log p(\boldsymbol{x}).
$$

This is exactly the familiar variational form

$$
\operatorname*{arg\,min}_\boldsymbol{x}
\mathcal{D}(K\boldsymbol{x},\boldsymbol{y}^\delta)+\mathcal{R}(\boldsymbol{x}),
$$

with a learned regularizer

$$
\mathcal{R}(\boldsymbol{x}) = -\log p(\boldsymbol{x}).
$$

This identity is worth highlighting. It shows that generative inverse problems are not disconnected from classical regularization theory. They are a modern way of learning the prior instead of hand-designing it.


## Generative Priors Through Latent Models and Denoisers

```{note}
This chapter is where the whole module closes conceptually: **inverse problems**, **probabilistic modeling**, and **learned reconstruction algorithms** meet in a single mathematical framework.
```

Suppose that a generative model provides a decoder or generator

$$
G_{\boldsymbol{\Theta}} : \mathbb{R}^k \to \mathbb{R}^n,
\qquad
\boldsymbol{x} = G_{\boldsymbol{\Theta}}(\boldsymbol{z}),
$$

with latent prior

$$
\boldsymbol{z} \sim \mathcal{N}(0,I).
$$

Then one may solve the inverse problem in latent space rather than in image space:

$$
\widehat{\boldsymbol{z}}
=
\operatorname*{arg\,min}_{\boldsymbol{z}}
\frac{1}{2\sigma^2}\|KG_{\boldsymbol{\Theta}}(\boldsymbol{z})-\boldsymbol{y}^\delta\|_2^2
+
\frac{\lambda}{2}\|\boldsymbol{z}\|_2^2.
$$

The reconstruction is then

$$
\widehat{\boldsymbol{x}}=G_{\boldsymbol{\Theta}}(\widehat{\boldsymbol{z}}).
$$

This strategy is mathematically appealing because the search space has dimension $k$ instead of $n$, often with $k \ll n$, as already emphasized in generative compressed sensing methods such as {cite}`bora2017compressed`. The generator acts as a nonlinear low-dimensional manifold of plausible images.


In [ ]:
# Higher-dimensional latent optimization on an image patch.
from pathlib import Path

def course_asset_path(name):
    here = Path.cwd().resolve()
    for base in (here, here.parent, here.parent.parent):
        candidate = base / 'imgs' / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not locate imgs/{name} from {here}')
from PIL import Image
import numpy as np
import torch

torch.manual_seed(0)

img = Image.open(course_asset_path('Mayo.png')).convert('L').resize((16, 16))
x_true = torch.tensor(np.array(img), dtype=torch.float32).reshape(-1) / 255.0

latent_dim = 12
image_dim = x_true.numel()
G = torch.randn(image_dim, latent_dim) / latent_dim**0.5
A = torch.randn(80, image_dim) / image_dim**0.5

# Project the true patch approximately onto the generator range.
z_ref = torch.linalg.lstsq(G, x_true.unsqueeze(1)).solution.squeeze(1)
x_range = G @ z_ref
y = A @ x_range

z = torch.zeros(latent_dim, requires_grad=True)
optimizer = torch.optim.Adam([z], lr=5e-2)

for step in range(200):
    x_hat = G @ z
    loss = torch.mean((A @ x_hat - y) ** 2) + 1e-3 * torch.mean(z ** 2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step in [0, 49, 99, 199]:
        image_rmse = torch.mean((x_hat.detach() - x_range) ** 2).sqrt().item()
        print(f'Step {step + 1:03d} | measurement loss = {loss.item():.6f} | image RMSE = {image_rmse:.6f}')

print('Reference latent norm:', float(torch.norm(z_ref)))
print('Recovered latent norm:', float(torch.norm(z.detach())))


Compared with the earlier scalar example, this version is closer to the real logic of latent reconstruction. The unknown object now lives in a higher-dimensional image space, but the optimization still takes place in a lower-dimensional latent space.


In [ ]:
import torch

torch.manual_seed(0)

def generator(z):
    return torch.stack([z, z**2], dim=-1)

A = torch.tensor([[1.0, -0.5]])
z_true = torch.tensor(0.8)
x_true = generator(z_true)
y = (A @ x_true).squeeze()

z = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.Adam([z], lr=0.1)

for _ in range(200):
    x = generator(z)
    loss = ((A @ x).squeeze() - y) ** 2 + 0.05 * z**2
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print('True latent variable:', float(z_true))
print('Recovered latent variable:', float(z.detach()))
print('True image:', x_true)
print('Recovered image:', generator(z.detach()))


**Strengths and limitations of latent-space methods.**

The strengths are clear:

- strong regularization through dimensionality reduction;
- explicit control through the latent prior;
- compatibility with both VAEs and **GAN** generators.

But the limitations are equally important:

- the optimization problem in latent space is nonconvex;
- the result depends strongly on how expressive the generator range is;
- if the true image lies outside the learned generator manifold, the reconstruction is biased no matter how informative the data are.

This last point should be stressed in class. A generative prior is powerful only to the extent that it covers the image class of interest.


**Plug-and-play and denoiser priors.**

A second strategy does not parameterize the solution explicitly as $\boldsymbol{x}=G_{\boldsymbol{\Theta}}(\boldsymbol{z})$. Instead, it uses a learned denoiser as a prior step inside an iterative algorithm, following the plug-and-play (PnP) framework {cite}`venkatakrishnan2013plug`.

Start from a variational problem

$$
\min_\boldsymbol{x}
\frac{1}{2\sigma^2}\|K\boldsymbol{x}-\boldsymbol{y}^\delta\|_2^2+\lambda R(\boldsymbol{x}).
$$

If one applies a splitting scheme such as HQS or ADMM, the iteration alternates between:

- a data-consistency update, which uses the forward model;
- a prior update, which can be interpreted as denoising.

Schematically, one writes

$$
\boldsymbol{x}^{k+1}\approx D_\tau(\boldsymbol{z}^k),
$$

where $D_\tau$ is a learned denoiser with effective noise level $\tau$.

This framework is attractive because it preserves a visible role for the forward operator while allowing the prior to be highly expressive.

In [ ]:
# Toy HQS-like iteration with a simple denoiser surrogate.
from pathlib import Path

def course_asset_path(name):
    here = Path.cwd().resolve()
    for base in (here, here.parent, here.parent.parent):
        candidate = base / 'imgs' / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not locate imgs/{name} from {here}')
from PIL import Image
import numpy as np
import torch

def smooth(v):
    kernel = torch.tensor([[1.0, 1.0, 1.0], [1.0, 2.0, 1.0], [1.0, 1.0, 1.0]], dtype=torch.float32)
    kernel = (kernel / kernel.sum()).view(1, 1, 3, 3)
    return torch.nn.functional.conv2d(v, kernel, padding=1)

img = Image.open(course_asset_path('GoPro.jpg')).convert('L').resize((32, 32))
x_true = torch.tensor(np.array(img), dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0
y = (x_true + 0.08 * torch.randn_like(x_true)).clamp(0.0, 1.0)

x = y.clone()
for k in range(5):
    z = 0.6 * y + 0.4 * x
    x = smooth(z)
    rmse = torch.mean((x - x_true) ** 2).sqrt().item()
    print(f'Iteration {k + 1}: RMSE = {rmse:.6f}')


This is not a true **plug-and-play** denoiser learned from data, but it mirrors the algorithmic structure: alternate a data-oriented update with a prior-oriented denoising step. In later, more advanced versions of the course, this placeholder denoiser can be replaced by a pretrained neural model.


## Diffusion Models and Posterior Sampling

Diffusion and **score-based** models offer an even richer possibility. Instead of only providing a denoising operator, they approximate the prior score

$$
\nabla_\boldsymbol{x} \log p(\boldsymbol{x}),
$$

or its noisy-scale analogues. This is crucial because the posterior score satisfies

$$
\nabla_\boldsymbol{x} \log p(\boldsymbol{x}| \boldsymbol{y}^\delta)
=
\nabla_\boldsymbol{x} \log p(\boldsymbol{y}^\delta| \boldsymbol{x})
+
\nabla_\boldsymbol{x} \log p(\boldsymbol{x}).
$$

For **Gaussian noise**, the likelihood score is explicit:

$$
\nabla_\boldsymbol{x} \log p(\boldsymbol{y}^\delta| \boldsymbol{x})
=
-\frac{1}{\sigma^2}K^T(K\boldsymbol{x}-\boldsymbol{y}^\delta).
$$

Hence, if a generative model supplies the prior score, then posterior-guided sampling becomes possible. This is conceptually different from a pure MAP estimate: the goal is no longer only to find a single reconstruction, but potentially to sample several plausible reconstructions from an approximation of the posterior distribution.


In [ ]:
# Tiny posterior correction step combining a prior-like score and a likelihood gradient.
import torch

torch.manual_seed(0)

A = torch.tensor([[1.0, 0.0], [0.0, 0.5]])
y = torch.tensor([0.8, -0.2])
x = torch.tensor([1.5, -1.0])
step = 0.2
sigma2 = 0.1

prior_score = -x
likelihood_score = -(A.T @ (A @ x - y)) / sigma2
posterior_update = x + step * (prior_score + likelihood_score)

print('Current state:', x)
print('Prior score:', prior_score)
print('Likelihood score:', likelihood_score)
print('One posterior-guided update:', posterior_update)


## A Useful Mental Model for Diffusion Inverse Solvers

Most diffusion inverse-problem algorithms can be explained through the same recurring pattern. At time step $t$ one has a current sample $\boldsymbol{x}_t$. The method then:

1. uses the diffusion model to produce a prior-informed estimate, often through a predicted clean image $\widehat{\boldsymbol{x}}_0$ or a score estimate;
2. uses the forward operator and the measurement to compute a data-consistency correction;
3. combines both pieces into the next iterate $\boldsymbol{x}_{t-1}$.

The differences between methods such as **DPS**, **DiffPIR**, and **DDNM** lie mainly in how this data-consistency correction is computed and how strongly it is enforced.


## DPS: Diffusion Posterior Sampling

A representative posterior-sampling method is **DPS** {cite}`chung2023diffusion`. The guiding idea is to keep the diffusion reverse process intact as the prior mechanism, but to add a correction driven by the likelihood at each reverse step.

At a schematic level, one may view a DPS-like update as

$$
\boldsymbol{x}_{t-1}
\approx
\text{prior-step}(\boldsymbol{x}_t)
-\eta_t K^T(K\widehat{\boldsymbol{x}}_0(\boldsymbol{x}_t)-\boldsymbol{y}^\delta),
$$

where $\widehat{\boldsymbol{x}}_0(\boldsymbol{x}_t)$ is the current estimate of the clean image extracted from the diffusion model.

Its main strength is flexibility: it can handle general noisy inverse problems and even nonlinear forward models. Its main weakness is that the correction is approximate and hyperparameter-sensitive.


In [ ]:
# Toy DPS-like sampler for a linear inverse problem.
import torch

torch.manual_seed(0)

A = torch.tensor([[1.0, 0.0], [0.4, 1.0]])
y = torch.tensor([0.5, -0.1])
sigma2 = 0.05
x = torch.randn(2)

for t in reversed(range(5)):
    x0_hat = 0.8 * x
    prior_step = 0.9 * x0_hat + 0.1 * torch.randn_like(x)
    grad_like = A.T @ (A @ x0_hat - y) / sigma2
    x = prior_step - 0.02 * grad_like
    print(f't = {t} | state = {x.tolist()}')


## DiffPIR: Diffusion in a Plug-and-Play Style

**DiffPIR** {cite}`zhu2023diffpir` can be understood as a bridge between diffusion sampling and classical plug-and-play optimization. Instead of using a direct likelihood-gradient correction as in DPS, it is often presented in a **proximal** or **HQS-like** form.

The practical logic is the following:

1. use the diffusion model at time $t$ to estimate a clean image $\widehat{\boldsymbol{x}}_0^t$;
2. apply a data-fidelity update or proximal step to obtain a corrected image estimate;
3. map this corrected estimate back into the diffusion update formula for the next reverse step.

This makes DiffPIR attractive when one wants to keep a more optimization-oriented interpretation of the inverse solver.


In [ ]:
# Toy DiffPIR-like update with a proximal data-consistency step.
import torch

torch.manual_seed(0)

def prox_data(x0_hat, y, lam):
    return (x0_hat + lam * y) / (1.0 + lam)

y = torch.tensor([0.3, -0.7])
x_t = torch.randn(2)

for t in reversed(range(5)):
    x0_hat = 0.85 * x_t
    x0_data = prox_data(x0_hat, y, lam=4.0)
    eps_hat = x_t - x0_data
    if t > 0:
        x_t = x0_data + 0.15 * eps_hat + 0.05 * torch.randn_like(x_t)
    else:
        x_t = x0_data
    print(f't = {t} | corrected state = {x_t.tolist()}')


## DDNM: Null-Space Guidance for Linear Problems

**DDNM** {cite}`wang2022ddnm` is built around a sharper geometric idea. For a linear operator $K$, one can decompose the image space into a **range** component that is constrained by the measurements and a **null-space** component that is not directly observed.

The key intuition is then:

- enforce measurement consistency on the observed component;
- let the diffusion model restore or sample the missing null-space component.

This is particularly natural for linear degradation models such as inpainting, deblurring, compressed sensing, and super-resolution. Compared with DPS, DDNM tends to impose stronger consistency for the part of the signal that is determined by the measurements. Compared with DiffPIR, its formulation is more explicitly tied to linear algebra and null-space structure.


In [ ]:
# Toy DDNM-like range/null-space splitting.
import torch

torch.manual_seed(0)

A = torch.tensor([[1.0, 0.0]])
y = torch.tensor([0.6])
range_part = torch.tensor([y.item(), 0.0])
null_basis = torch.tensor([0.0, 1.0])

x = torch.randn(2)
for t in reversed(range(5)):
    null_coeff = torch.dot(x, null_basis)
    null_coeff = 0.8 * null_coeff + 0.2 * torch.randn(())
    x = range_part + null_coeff * null_basis
    print(f't = {t} | data-consistent state = {x.tolist()}')


## Comparing DPS, DiffPIR, and DDNM

A compact way to compare them is the following.

- **DPS:** posterior sampling through a likelihood gradient correction added to the reverse diffusion dynamics. Flexible and broadly applicable, but approximate and sensitive to tuning.
- **DiffPIR:** diffusion embedded inside a plug-and-play / proximal restoration loop. More optimization-flavored and often easier to interpret as iterative reconstruction.
- **DDNM:** null-space based correction for linear inverse problems. Especially elegant when the operator structure is simple and strong measurement consistency can be enforced directly.

In practice, these methods are not mutually exclusive philosophical camps. They form a spectrum between **probabilistic reverse sampling**, **optimization-inspired reconstruction**, and **operator-aware consistency enforcement**.


## Limitations and Points of Attention

A strong generative prior can produce highly realistic images, but realism is not the same as truth. In scientific and medical imaging, this distinction is critical.

The main points of attention are the following.

- **Hallucinations.** The method may generate visually plausible structures that are weakly supported by the data.
- **Operator mismatch.** If the forward operator used at test time differs from the one assumed by the algorithm, posterior guidance can become misleading.
- **Noise-model mismatch.** The likelihood term depends on the noise model. If this is incorrect, data consistency is not being enforced in the right way.
- **Sampling cost.** Posterior diffusion methods are often expensive because they require many reverse steps, each involving both a neural-network evaluation and forward/adjoint computations.
- **Hyperparameter sensitivity.** Step sizes, noise schedules, proximal weights, and correction strengths all matter.
- **Uncertainty interpretation.** Producing many samples is not automatically the same thing as calibrated uncertainty quantification.

Every generative inverse method should be judged along at least two axes, **prior plausibility** and **data fidelity**.


```{warning}
In medical or scientific imaging, a plausible detail that is unsupported by the data can be more dangerous than a visibly imperfect reconstruction. This is why learned priors must always be discussed together with data fidelity, operator assumptions, and uncertainty.
```


## Summary

The final conceptual map should be:

- Bayesian inverse problems combine likelihood and prior information;
- latent generative models restrict the search space to a learned manifold;
- plug-and-play methods use learned denoisers inside iterative reconstruction;
- diffusion priors enable posterior-aware sampling rather than only point estimation;
- DPS, DiffPIR, and DDNM differ mainly in how they inject data consistency into the generative process;
- generative priors are powerful, but they must always be constrained by the physics of the measurements.


## Exercises

1. Starting from Bayes' theorem, derive the MAP variational objective for **Gaussian noise**.
2. Explain the main benefit and the main limitation of latent-space reconstruction.
3. Describe the difference between a MAP estimate and **posterior sampling**.
4. In your own words, what is the main algorithmic difference between DPS and DiffPIR?
5. Why is DDNM especially natural for linear inverse problems?
6. Why can a strong generative prior also increase the risk of **hallucinations**?
7. Code exercise: modify one of the toy DPS / DiffPIR / DDNM updates and observe how the state changes when the data-consistency strength is increased.


## Further Reading

The Bayesian probabilistic perspective on inverse problems is developed rigorously in {cite}`kaipio2005statistical`. Latent-space reconstruction with generative priors is explored in {cite}`bora2017compressed`. Plug-and-play priors originate in {cite}`venkatakrishnan2013plug`. Diffusion posterior sampling is described in {cite}`chung2023diffusion`; the plug-and-play diffusion restoration viewpoint is developed in {cite}`zhu2023diffpir`; and null-space diffusion restoration is presented in {cite}`wang2022ddnm`.

A useful study strategy is to classify each method by three questions: how is the prior represented, how is data fidelity incorporated, and does the algorithm return a single estimate or a distribution of plausible reconstructions?
